In [ ]:
!pip install textblob

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from textblob import TextBlob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Amazon_Extern/glassdoor_cleaned_processed.csv')

In [ ]:
df.columns

In [ ]:
df['sentiment_polarity'] = df['cleaned_text'].apply(lambda x: TextBlob(x).sentiment.polarity)
df['sentiment_subjectivity'] = df['cleaned_text'].apply(lambda x: TextBlob(x).sentiment.subjectivity)

In [ ]:
columns_to_merge = ['review_pros_cleaned', 'review_cons_cleaned', 'advice_to_management_cleaned', 'summary_cleaned']

# Fill NaN values in specified columns with empty strings to allow concatenation
for col in columns_to_merge:
    if col in df.columns:
        df[col] = df[col].fillna('')
    else:
        print(f"Warning: Column '{col}' not found in DataFrame. Skipping this column for merging.")

# Create the 'merged_all' column by concatenating the cleaned text columns
# Use .astype(str) just in case there are non-string types after fillna (though fillna('') should make them string-compatible)
df['merged_all'] = df[columns_to_merge].apply(lambda x: ' '.join(x.astype(str)), axis=1)

# Perform sentiment analysis on the 'merged_all' column as requested

print("Merged columns into 'merged_all' and performed sentiment analysis.")
print(df[['merged_all', 'sentiment_polarity', 'sentiment_subjectivity']].head())

In [ ]:
df['sentiment_polarity'] = df['merged_all'].apply(lambda x: TextBlob(x).sentiment.polarity)
df['sentiment_subjectivity'] = df['merged_all'].apply(lambda x: TextBlob(x).sentiment.subjectivity)

In [ ]:
df.head(2)

In [ ]:
def label_sentiment(score):
    if score > 0.1:
        return 'positive'
    elif score < -0.1:
        return 'negative'
    else:
        return 'neutral'

df['sentiment_label'] = df['sentiment_polarity'].apply(label_sentiment)
df.head(2)

In [ ]:
# Set a professional and minimal plot style with a white background
sns.set_theme(style='white')

# Prepare data for plotting
sentiment_counts = df['sentiment_label'].value_counts().reset_index()
sentiment_counts.columns = ['Sentiment', 'Count']

# Define a custom light blue palette for the sentiment labels
# We'll generate 3 shades of 'steelblue' (a medium-dark blue) and reverse it
# to get lighter shades first, then map them to sentiment for consistent appearance.
light_blue_palette_colors = sns.light_palette("steelblue", n_colors=3, reverse=True)

# Explicitly map sentiment labels to specific shades of blue for consistency
sentiment_colors_map = {
    'positive': light_blue_palette_colors[0], # Lightest blue
    'neutral': light_blue_palette_colors[1], # Medium blue
    'negative': light_blue_palette_colors[2]  # Darkest blue
}

# Ensure the order of sentiments for consistent bar placement and color mapping
sentiment_order = ['positive', 'neutral', 'negative']

# Create the bar chart using seaborn
plt.figure(figsize=(8, 6))
ax = sns.barplot(x='Sentiment', y='Count', data=sentiment_counts,
                 palette=sentiment_colors_map, # Use the custom light blue palette
                 order=sentiment_order,        # Ensure bars are in a specific order
                 hue='Sentiment', legend=False) # Use hue for color mapping, but hide the legend as it's redundant

plt.title('Distribution of Sentiment Labels', fontsize=16)
plt.xlabel('Sentiment Category', fontsize=12)
plt.ylabel('') # Remove Y-label as requested for a minimalist look
plt.xticks(rotation=0) # Keep labels horizontal for cleanliness
plt.grid(axis='y', linestyle='--', alpha=0.7) # Subtle grid on y-axis

# Add labels (counts) on top of the bars
for container in ax.containers:
    ax.bar_label(container, fmt='%d') # Format as integer

plt.tight_layout()
plt.show()

In [ ]:
# Set a professional and minimal plot style with a white background
sns.set_theme(style='white')

# Define a custom light blue palette for the sentiment labels
# We'll generate 3 shades of 'steelblue' (a medium-dark blue) and reverse it
# to get lighter shades first, then map them to sentiment for consistent appearance.
light_blue_palette_colors = sns.light_palette("steelblue", n_colors=3, reverse=True)

# Explicitly map sentiment labels to specific shades of blue for consistency
sentiment_colors_map = {
    'positive': light_blue_palette_colors[0], # Lightest blue
    'neutral': light_blue_palette_colors[1], # Medium blue
    'negative': light_blue_palette_colors[2]  # Darkest blue
}

# Order the sentiment counts to ensure consistent color mapping
sentiment_counts_ordered = df['sentiment_label'].value_counts().reindex(['positive', 'neutral', 'negative']).fillna(0)

# Get colors in the desired order for the pie chart
pie_colors = [sentiment_colors_map[label] for label in sentiment_counts_ordered.index]

sentiment_counts_ordered.plot(
    kind='pie',
    autopct='%1.0f%%',
    title="Sentiment Breakdown",
    colors=pie_colors, # Apply the custom light blue palette
    startangle=90
)
plt.ylabel('') # Keep the y-label removed for a cleaner look
plt.show()

In [ ]:
df.to_csv('/content/drive/MyDrive/Glassdoor_sentiment.csv', index=False)